# PatchCore ViT-B/16 Two-Block (`x224`)

This notebook is the canonical evaluation notebook for the two-block ViT-B/16 PatchCore experiment.

Default behavior:
- `RUN_TRAINING = False`
- load `artifacts/checkpoints/model.pt`
- load `artifacts/results/scores.npz`
- regenerate evaluation outputs in the local `artifacts/` folder

Outputs written by this notebook:
- `artifacts/checkpoints/model.pt`
- `artifacts/results/scores.npz`
- `artifacts/results/evaluation_metrics.json`
- `artifacts/results/test_scores.csv`
- `artifacts/results/test_defect_recall.csv`
- `artifacts/results/umap_test_embeddings.csv`
- `artifacts/plots/confusion_matrix_test.png`
- `artifacts/plots/score_distribution_test.png`
- `artifacts/plots/threshold_selection_tune.png`
- `artifacts/plots/umap_test_embeddings.png`


## Setup

Install optional notebook dependencies if needed.


In [ ]:
# -- 0. Install dependencies if needed -----------------------------------------
import importlib, subprocess, sys
for pkg in ['timm', 'tqdm']:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('timm + tqdm ready')

## Imports


In [ ]:
# -- 1. Imports ----------------------------------------------------------------
import os, gc, random, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
from tqdm.auto import tqdm
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')

print('Device:', DEVICE)
if USE_CUDA:
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB')

## Configuration


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/.')

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

# Configuration
DATA_PATH = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
IMAGE_SIZE = 224

# Optimized split from: 147,431 normal, 25,519 defect
# Train: 68% (normal only) | Tune: 14% (both classes) | Test: 18% (anomaly-realistic)
TRAIN_NORMAL_N = 40_000
TUNE_NORMAL_N = 5_000
TUNE_DEFECT_N = 5_000
TEST_NORMAL_N = 5_000
TEST_DEFECT_N = 250

# ViT settings
VIT_FEATURE_BLOCKS = (6, 8)
PATCH_EMBED_DIM = 256

# Memory bank
MEMORY_BANK_MAX_PATCHES = 600_000

# Scoring
SCORE_CHUNK = 512
PATCHCORE_NN_K = 3
TOPK_PATCH_RATIO = 0.03

# DataLoader
BATCH_SIZE = 128
NUM_WORKERS = 0

# Threshold sweep
THRESHOLD_PERCENTILE_MIN = 90
THRESHOLD_PERCENTILE_MAX = 99.9
THRESHOLD_PERCENTILE_STEPS = 100
THRESHOLD_GRID_STEPS = 300

# Outputs
ARTIFACT_DIR = str(PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/vit_b16/x224/two_block/artifacts')
CHECKPOINTS_DIR = os.path.join(ARTIFACT_DIR, 'checkpoints')
PLOTS_DIR = os.path.join(ARTIFACT_DIR, 'plots')
RESULTS_DIR = os.path.join(ARTIFACT_DIR, 'results')
MODEL_EXPORT_PATH = os.path.join(CHECKPOINTS_DIR, 'model.pt')
SCORES_EXPORT_PATH = os.path.join(RESULTS_DIR, 'scores.npz')
METRICS_EXPORT_PATH = os.path.join(RESULTS_DIR, 'evaluation_metrics.json')
TEST_SCORES_CSV_PATH = os.path.join(RESULTS_DIR, 'test_scores.csv')
DEFECT_RECALL_CSV_PATH = os.path.join(RESULTS_DIR, 'test_defect_recall.csv')
CLASSIFICATION_REPORT_PATH = os.path.join(RESULTS_DIR, 'classification_report.txt')
THRESHOLD_SELECTION_PNG_PATH = os.path.join(PLOTS_DIR, 'threshold_selection_tune.png')
CONFUSION_MATRIX_PNG_PATH = os.path.join(PLOTS_DIR, 'confusion_matrix_test.png')
SCORE_DISTRIBUTION_PNG_PATH = os.path.join(PLOTS_DIR, 'score_distribution_test.png')
UMAP_PNG_PATH = os.path.join(PLOTS_DIR, 'umap_test_embeddings.png')
UMAP_CSV_PATH = os.path.join(RESULTS_DIR, 'umap_test_embeddings.csv')
RUN_TRAINING = False

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

MODEL_ARTIFACT_EXISTS = os.path.exists(MODEL_EXPORT_PATH)
SCORES_ARTIFACT_EXISTS = os.path.exists(SCORES_EXPORT_PATH)

print('Artifacts directory:', ARTIFACT_DIR)
print(f'RUN_TRAINING={RUN_TRAINING}')
print(f'ViT blocks={VIT_FEATURE_BLOCKS}  embed_dim={PATCH_EMBED_DIM}')
print(f'Bank cap={MEMORY_BANK_MAX_PATCHES:,}  batch={BATCH_SIZE}  workers={NUM_WORKERS}')

if not RUN_TRAINING and not MODEL_ARTIFACT_EXISTS:
    raise FileNotFoundError(
        f'Saved model artifact not found at {MODEL_EXPORT_PATH}. '
    )

if not RUN_TRAINING and not SCORES_ARTIFACT_EXISTS:
    raise FileNotFoundError(
        f'Saved score bundle not found at {SCORES_EXPORT_PATH}. '
    )


## Load Dataset


In [ ]:
# -- 3. Load and clean dataset ------------------------------------------------
# Only DataFrames are created here - no tensors and no float32 copies.
# Raw wafer maps stay as object-dtype numpy arrays in the DataFrame.

from wafer_defect.data.legacy_pickle import read_legacy_pickle

df = read_legacy_pickle(DATA_PATH)
print('Raw shape:', df.shape)

def parse_failure_label(v):
    if v is None: return 'unknown'
    if isinstance(v, float) and np.isnan(v): return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)

df = df.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()

invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)

normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()

print(f'Labeled: {len(df):,}   Normal: {len(normal_df):,}   Defect: {len(defect_df):,}')
print()
print('Defect breakdown:')
print(defect_df['failure_label'].value_counts())


## Split


In [ ]:
# -- 4. Split -----------------------------------------------------------------
req_n = TRAIN_NORMAL_N + TUNE_NORMAL_N + TEST_NORMAL_N
req_d = TUNE_DEFECT_N + TEST_DEFECT_N

if len(normal_df) < req_n:
    raise ValueError(f'Need {req_n:,} normals, have {len(normal_df):,}')
if len(defect_df) < req_d:
    raise ValueError(f'Need {req_d:,} defects, have {len(defect_df):,}')

rng = np.random.default_rng(SEED)
ns = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)

a = TRAIN_NORMAL_N
b = a + TUNE_NORMAL_N
c = b + TEST_NORMAL_N

train_normal_df = ns.iloc[0:a].copy()
tune_normal_df = ns.iloc[a:b].copy()
test_normal_df = ns.iloc[b:c].copy()
tune_defect_df = ds.iloc[0:TUNE_DEFECT_N].copy()
test_defect_df = ds.iloc[TUNE_DEFECT_N:TUNE_DEFECT_N + TEST_DEFECT_N].copy()

# Free the large unsplit dataframes and keep only the slices.
del df, normal_df, defect_df, ns, ds
gc.collect()

print(f'Train normal : {len(train_normal_df):>7,}')
print(f'Tune  normal : {len(tune_normal_df):>7,}')
print(f'Tune  defect : {len(tune_defect_df):>7,}')
print(f'Test  normal : {len(test_normal_df):>7,}')
print(f'Test  defect : {len(test_defect_df):>7,}')


## Dataset


In [ ]:
# -- 5. Lazy WaferDataset -----------------------------------------------------
# No tensor is created until __getitem__ is called for a single sample.
# The DataLoader batches these calls across NUM_WORKERS processes,
# so at most BATCH_SIZE tensors exist in memory at once.
# Raw map storage: 1 wafer map ~ 26x26 int8 ~ 700 bytes
# vs tensor:       1 wafer map ~ 224x224x3 float32 ~ 602 KB
# Ratio: about 860x more compact in the DataFrame.

class WaferDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, size: int = 224):
        self.maps = frame['waferMap'].values
        self.labels = frame['is_anomaly'].values.astype(np.int64)
        self.size = size

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
        x = torch.tensor(arr, dtype=torch.long)
        x = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()
        x = F.interpolate(
            x.unsqueeze(0),
            size=(self.size, self.size),
            mode='nearest',
        ).squeeze(0)
        return x, int(self.labels[idx])


loader_kw = dict(
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_CUDA,
    persistent_workers=(NUM_WORKERS > 0),
)

train_loader = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
tune_normal_loader = DataLoader(WaferDataset(tune_normal_df, IMAGE_SIZE), **loader_kw)
tune_defect_loader = DataLoader(WaferDataset(tune_defect_df, IMAGE_SIZE), **loader_kw)
test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **loader_kw)
test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **loader_kw)

for name, ldr in [
    ('train_normal', train_loader),
    ('tune_normal', tune_normal_loader),
    ('tune_defect', tune_defect_loader),
    ('test_normal', test_normal_loader),
    ('test_defect', test_defect_loader),
]:
    print(f'{name:<14}: {len(ldr):>4} batches')

xb, yb = next(iter(train_loader))
print()
print(f'Batch shape: {tuple(xb.shape)}  dtype={xb.dtype}  sample labels={yb[:4].tolist()}')


## Define Model


In [ ]:
# -- 6. ViT-B/16 feature extractor -------------------------------------------
# Forward hooks capture token outputs from two blocks.
# We drop CLS from each and concatenate them into [B, 196, 1536].
# The projection layer maps them to [B, 196, PATCH_EMBED_DIM].

class ViTPatchExtractor(nn.Module):
    def __init__(self, block_indices=VIT_FEATURE_BLOCKS, proj_dim: int = PATCH_EMBED_DIM):
        super().__init__()
        self.vit = timm.create_model(
            'vit_base_patch16_224.augreg_in21k_ft_in1k',
            pretrained=True,
            num_classes=0,
        )
        self.block_indices = tuple(block_indices)
        self._feat = {}

        for bi in self.block_indices:
            self.vit.blocks[bi].register_forward_hook(
                lambda m, i, o, bi=bi: self._feat.__setitem__(bi, o)
            )

        self.concat_dim = 768 * len(self.block_indices)
        self.proj = nn.Linear(self.concat_dim, proj_dim, bias=False)

    def forward(self, x):
        self._feat = {}
        self.vit(x)
        toks = [self._feat[bi][:, 1:, :] for bi in self.block_indices]
        return torch.cat(toks, dim=-1)


extractor = ViTPatchExtractor().to(DEVICE).eval()
for p in extractor.parameters():
    p.requires_grad = False

with torch.inference_mode():
    dummy = torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    out = extractor(dummy)
    proj = extractor.proj(out)
print(f'ViT blocks-{VIT_FEATURE_BLOCKS} output : {tuple(out.shape)}')
print(f'After projection         : {tuple(proj.shape)}')


## Train Or Load Artifact


In [ ]:
# -- 7. Build memory bank or load saved model ---------------------------------

def extract_embeddings(xb: torch.Tensor) -> torch.Tensor:
    """L2-normalized embeddings with shape [B * 196, proj_dim] on the active device."""
    with torch.inference_mode():
        with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
            feat = extractor(xb)
            emb = extractor.proj(feat)
        emb = emb.float().reshape(-1, emb.shape[-1])
        emb = F.normalize(emb, p=2, dim=1)
    return emb

if RUN_TRAINING:
    sampled = []
    total_seen = 0
    sample_ratio = None

    print('Building memory bank..')
    bank_iter = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc='Bank build',
        unit='batch',
    )
    for step, (xb, _) in bank_iter:
        xb = xb.to(DEVICE)
        emb = extract_embeddings(xb)
        total_seen += len(emb)

        if sample_ratio is None:
            tokens_per_img = len(emb) // len(xb)
            estimated_total = tokens_per_img * len(train_normal_df)
            sample_ratio = min(1.0, MEMORY_BANK_MAX_PATCHES / estimated_total)
            print(f'  Tokens/image : {tokens_per_img}')
            print(f'  Est. total   : {estimated_total:,}')
            print(f'  Sample ratio : {sample_ratio:.5f}')

        if sample_ratio < 1.0:
            keep_n = max(1, int(round(len(emb) * sample_ratio)))
            keep_idx = torch.randperm(len(emb), device=DEVICE)[:keep_n]
            emb = emb[keep_idx]

        sampled.append(emb)

        if (step + 1) % 20 == 0 or (step + 1) == len(train_loader):
            sampled_total = sum(len(chunk) for chunk in sampled)
            bank_iter.set_postfix(bank_tokens=f'{sampled_total:,}')

    memory_bank = torch.cat(sampled, dim=0)
    del sampled
    gc.collect()

    if len(memory_bank) > MEMORY_BANK_MAX_PATCHES:
        keep_idx = torch.randperm(len(memory_bank), device=DEVICE)[:MEMORY_BANK_MAX_PATCHES]
        memory_bank = memory_bank[keep_idx]

    memory_bank = F.normalize(memory_bank, p=2, dim=1).contiguous()
    memory_bank_t = memory_bank.t().contiguous()

    mb_mb = memory_bank.element_size() * memory_bank.numel() / 1e6
    print(f'Final bank : {len(memory_bank):,} x {memory_bank.shape[1]}-d ({mb_mb:.1f} MB VRAM)')
else:
    print(f'Loading saved model artifact from: {MODEL_EXPORT_PATH}')
    saved_artifact = torch.load(MODEL_EXPORT_PATH, map_location='cpu')
    if 'extractor_state_dict' not in saved_artifact:
        raise KeyError(f'Missing extractor_state_dict in {MODEL_EXPORT_PATH}')
    extractor.load_state_dict(saved_artifact['extractor_state_dict'])
    print('Loaded saved extractor weights for evaluation-only mode.')


## Score Bundle


In [ ]:
# -- 8. Generate or load score bundle ------------------------------------------

def min_dist_to_bank(patches, bank_t, chunk=512, k=3):
    distances = []
    for start in range(0, len(patches), chunk):
        patch_batch = patches[start:start + chunk]
        similarity = patch_batch @ bank_t
        neighbor_count = min(k, similarity.shape[1])
        best = similarity.topk(neighbor_count, dim=1).values
        dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best, min=0.0)).mean(dim=1)
        distances.append(dist)
    return torch.cat(distances)


def score_loader(loader, bank_t, topk_ratio=0.1, nn_k=3, desc='Scoring'):
    score_chunks = []
    with torch.inference_mode():
        progress = tqdm(loader, total=len(loader), desc=desc, unit='batch')
        for xb, _ in progress:
            xb = xb.to(DEVICE)
            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                feat = extractor(xb)
                emb = extractor.proj(feat)
            emb = emb.float().reshape(-1, emb.shape[-1])
            emb = F.normalize(emb, p=2, dim=1)

            patch_scores = min_dist_to_bank(emb, bank_t, SCORE_CHUNK, nn_k)
            batch_size = len(xb)
            patch_scores = patch_scores.reshape(batch_size, -1)
            topk = max(1, int(round(patch_scores.shape[1] * topk_ratio)))
            topk = min(topk, patch_scores.shape[1])
            image_scores = patch_scores.topk(topk, dim=1).values.mean(dim=1)
            score_chunks.append(image_scores.cpu())
    return torch.cat(score_chunks).numpy()

if RUN_TRAINING:
    score_kwargs = dict(topk_ratio=TOPK_PATCH_RATIO, nn_k=PATCHCORE_NN_K)

    train_scores = score_loader(train_loader, memory_bank_t, desc='Score train-normal', **score_kwargs)
    tune_normal_scores = score_loader(tune_normal_loader, memory_bank_t, desc='Score tune-normal', **score_kwargs)
    tune_defect_scores = score_loader(tune_defect_loader, memory_bank_t, desc='Score tune-defect', **score_kwargs)
    test_normal_scores = score_loader(test_normal_loader, memory_bank_t, desc='Score test-normal', **score_kwargs)
    test_defect_scores = score_loader(test_defect_loader, memory_bank_t, desc='Score test-defect', **score_kwargs)

    mu = float(np.mean(train_scores))
    std = float(np.std(train_scores) + 1e-8)

    def znorm(values):
        return (values - mu) / std

    train_scores_z = znorm(train_scores)
    tune_normal_scores_z = znorm(tune_normal_scores)
    tune_defect_scores_z = znorm(tune_defect_scores)
    test_normal_scores_z = znorm(test_normal_scores)
    test_defect_scores_z = znorm(test_defect_scores)

    np.savez_compressed(
        SCORES_EXPORT_PATH,
        train_scores_z=train_scores_z,
        tune_normal_scores_z=tune_normal_scores_z,
        tune_defect_scores_z=tune_defect_scores_z,
        test_normal_scores_z=test_normal_scores_z,
        test_defect_scores_z=test_defect_scores_z,
        train_mu=np.array(mu),
        train_std=np.array(std),
    )
    print(f'Saved score bundle: {SCORES_EXPORT_PATH}')
    print(f'Score normalization: mu={mu:.6f}  sigma={std:.6f}')
else:
    with np.load(SCORES_EXPORT_PATH) as score_bundle:
        train_scores_z = score_bundle['train_scores_z']
        tune_normal_scores_z = score_bundle['tune_normal_scores_z']
        tune_defect_scores_z = score_bundle['tune_defect_scores_z']
        test_normal_scores_z = score_bundle['test_normal_scores_z']
        test_defect_scores_z = score_bundle['test_defect_scores_z']
        mu = float(score_bundle['train_mu'])
        std = float(score_bundle['train_std'])
    print(f'Loaded saved score bundle from: {SCORES_EXPORT_PATH}')


## Threshold Selection


In [ ]:
# -- 9. Threshold selection -----------------------------------------------------

with np.load(SCORES_EXPORT_PATH) as score_bundle:
    tune_normal_scores_z = score_bundle['tune_normal_scores_z']
    tune_defect_scores_z = score_bundle['tune_defect_scores_z']
    test_normal_scores_z = score_bundle['test_normal_scores_z']
    test_defect_scores_z = score_bundle['test_defect_scores_z']
    mu = float(score_bundle['train_mu'])
    std = float(score_bundle['train_std'])

y_tune = np.concatenate([
    np.zeros(len(tune_normal_scores_z), dtype=int),
    np.ones(len(tune_defect_scores_z), dtype=int),
])
score_tune = np.concatenate([tune_normal_scores_z, tune_defect_scores_z])

grid_lo = float(np.percentile(score_tune, 1.0))
grid_hi = float(np.percentile(score_tune, 99.0))
threshold_grid = np.linspace(grid_lo, grid_hi, THRESHOLD_GRID_STEPS)

best_bal_acc = -1.0
best_f1 = -1.0
threshold_z = float(np.median(score_tune))
for threshold_candidate in threshold_grid:
    pred = (score_tune > threshold_candidate).astype(int)
    tp = int(((pred == 1) & (y_tune == 1)).sum())
    tn = int(((pred == 0) & (y_tune == 0)).sum())
    fp = int(((pred == 1) & (y_tune == 0)).sum())
    fn = int(((pred == 0) & (y_tune == 1)).sum())

    tpr = tp / max(tp + fn, 1)
    tnr = tn / max(tn + fp, 1)
    bal_acc = 0.5 * (tpr + tnr)
    precision = tp / max(tp + fp, 1)
    recall = tpr
    f1 = 2.0 * precision * recall / max(precision + recall, 1e-12)

    if (bal_acc > best_bal_acc) or (np.isclose(bal_acc, best_bal_acc) and f1 > best_f1):
        best_bal_acc = bal_acc
        best_f1 = f1
        threshold_z = float(threshold_candidate)

threshold_raw = mu + threshold_z * std
print(f'Selected z-threshold: {threshold_z:.4f}  raw: {threshold_raw:.6f}')
print(f'Tune balanced accuracy: {best_bal_acc:.4f}  tune F1: {best_f1:.4f}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(tune_normal_scores_z, bins=50, alpha=0.55, label='Tune normal', color='steelblue', density=True)
ax.hist(tune_defect_scores_z, bins=50, alpha=0.45, label='Tune defect', color='tomato', density=True)
ax.axvline(threshold_z, color='red', linewidth=2, linestyle='--', label=f'Threshold z={threshold_z:.3f}')
ax.set_xlabel('Anomaly score (z)')
ax.set_ylabel('Density')
ax.set_title('Score Distribution - Tune Split')
ax.legend()
fig.tight_layout()
fig.savefig(THRESHOLD_SELECTION_PNG_PATH, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved threshold selection plot: {THRESHOLD_SELECTION_PNG_PATH}')


## Test Evaluation


In [ ]:
# -- 10. Final test evaluation --------------------------------------------------

y_true = np.concatenate([
    np.zeros(len(test_normal_scores_z), dtype=int),
    np.ones(len(test_defect_scores_z), dtype=int),
])
scores = np.concatenate([test_normal_scores_z, test_defect_scores_z])
y_pred = (scores > threshold_z).astype(int)

roc_auc = float(roc_auc_score(y_true, scores))
report = classification_report(y_true, y_pred, target_names=['normal', 'anomaly'])
print(f'ROC-AUC : {roc_auc:.4f}')
print(f'Z-thr   : {threshold_z:.4f}   raw: {threshold_raw:.6f}')
print(report)

Path(CLASSIFICATION_REPORT_PATH).write_text(report, encoding='utf-8')
print(f'Saved classification report: {CLASSIFICATION_REPORT_PATH}')

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    cbar=False,
    ax=ax,
    xticklabels=['pred normal', 'pred anomaly'],
    yticklabels=['true normal', 'true anomaly'],
)
ax.set_title('Confusion Matrix (Test)')
fig.tight_layout()
fig.savefig(CONFUSION_MATRIX_PNG_PATH, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved confusion matrix: {CONFUSION_MATRIX_PNG_PATH}')

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.kdeplot(test_normal_scores_z, label='Normal', fill=True, alpha=0.35, ax=ax)
sns.kdeplot(test_defect_scores_z, label='Defect', fill=True, alpha=0.35, ax=ax)
ax.axvline(threshold_z, color='red', linestyle='--', label=f'z={threshold_z:.2f}')
ax.set_xlabel('Anomaly score (z)')
ax.set_ylabel('Density')
ax.set_title('Score Distribution (Test)')
ax.legend()
fig.tight_layout()
fig.savefig(SCORE_DISTRIBUTION_PNG_PATH, dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved score distribution plot: {SCORE_DISTRIBUTION_PNG_PATH}')

test_scores_df = pd.concat([
    test_normal_df[['failure_label', 'is_anomaly']].assign(split='test_normal'),
    test_defect_df[['failure_label', 'is_anomaly']].assign(split='test_defect'),
], ignore_index=True)
test_scores_df['score_z'] = scores
test_scores_df['predicted_anomaly'] = y_pred
test_scores_df.to_csv(TEST_SCORES_CSV_PATH, index=False)
print(f'Saved test scores: {TEST_SCORES_CSV_PATH}')

defect_recall_df = (
    test_scores_df.loc[test_scores_df['is_anomaly'] == 1]
  .groupby('failure_label')
  .agg(
        count=('predicted_anomaly', 'count'),
        detected=('predicted_anomaly', 'sum'),
        recall=('predicted_anomaly', 'mean'),
        mean_score=('score_z', 'mean'),
    )
  .reset_index()
  .sort_values(['recall', 'count', 'failure_label'], ascending=[True, False, True])
)
defect_recall_df.to_csv(DEFECT_RECALL_CSV_PATH, index=False)
print('Per-class defect recall:')
display(defect_recall_df.round(3))
print(f'Saved defect recall summary: {DEFECT_RECALL_CSV_PATH}')


## UMAP


In [ ]:
# -- 11. UMAP diagnostic -------------------------------------------------------

import sys
import subprocess
from sklearn.decomposition import PCA

try:
    import umap.umap_ as umap
except Exception:
    print('Installing umap-learn..')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
    import umap.umap_ as umap


def collect_image_embeddings(loader, frame, split_name, desc):
    embeddings = []
    metadata_rows = []
    seen = 0
    with torch.inference_mode():
        for xb, yb in tqdm(loader, total=len(loader), desc=desc, unit='batch'):
            batch_size = len(yb)
            xb = xb.to(DEVICE)
            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                feat = extractor(xb)
                emb = extractor.proj(feat)
            img_emb = F.normalize(emb.float().mean(dim=1), p=2, dim=1)
            embeddings.append(img_emb.cpu().numpy())

            batch_meta = frame.iloc[seen:seen + batch_size][['failure_label', 'is_anomaly']].copy()
            batch_meta = batch_meta.reset_index(drop=True)
            batch_meta['split'] = split_name
            metadata_rows.append(batch_meta)
            seen += batch_size

    X = np.concatenate(embeddings, axis=0)
    meta = pd.concat(metadata_rows, ignore_index=True)
    return X, meta

print('Collecting full test split embeddings for UMAP..')
X_normal, meta_normal = collect_image_embeddings(
    test_normal_loader,
    test_normal_df,
    split_name='test_normal',
    desc='Embed test-normal',
)
X_defect, meta_defect = collect_image_embeddings(
    test_defect_loader,
    test_defect_df,
    split_name='test_defect',
    desc='Embed test-defect',
)

X = np.concatenate([X_normal, X_defect], axis=0)
umap_df = pd.concat([meta_normal, meta_defect], ignore_index=True)
umap_df.insert(0, 'point_index', np.arange(len(umap_df)))
umap_df['score_z'] = np.concatenate([test_normal_scores_z, test_defect_scores_z])
umap_df['predicted_anomaly'] = (umap_df['score_z'] > threshold_z).astype(int)
umap_df['label_name'] = umap_df['is_anomaly'].map({0: 'normal', 1: 'defect'})

n_pca = min(32, X.shape[1], X.shape[0] - 1)
Xp = PCA(n_components=n_pca, random_state=SEED).fit_transform(X)

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=SEED,
    transform_seed=SEED,
    low_memory=True,
    verbose=True,
)
coords = reducer.fit_transform(Xp)

umap_df['umap_1'] = coords[:, 0]
umap_df['umap_2'] = coords[:, 1]

fig, ax = plt.subplots(figsize=(8, 6))
normal_mask = umap_df['is_anomaly'].to_numpy() == 0
defect_mask = ~normal_mask
ax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, label='Normal', c='steelblue', linewidths=0)
ax.scatter(coords[defect_mask, 0], coords[defect_mask, 1], s=8, alpha=0.60, label='Defect', c='tomato', linewidths=0)
ax.set_title('UMAP of Test Image Embeddings (ViT PatchCore)')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend()
fig.tight_layout()
fig.savefig(UMAP_PNG_PATH, dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

umap_df.to_csv(UMAP_CSV_PATH, index=False)
print(f'Saved UMAP figure: {UMAP_PNG_PATH}')
print(f'Saved UMAP points: {UMAP_CSV_PATH}')


## Save Outputs


In [ ]:
# -- 12. Save outputs ----------------------------------------------------------

artifact = {
    'extractor_state_dict': extractor.state_dict(),
    'threshold_z': float(threshold_z),
    'threshold_raw': float(threshold_raw),
    'train_score_mu': float(mu),
    'train_score_std': float(std),
    'config': {
        'backbone': 'vit_base_patch16_224.augreg_in21k_ft_in1k',
        'vit_feature_blocks': list(VIT_FEATURE_BLOCKS),
        'patch_embed_dim': PATCH_EMBED_DIM,
        'image_size': IMAGE_SIZE,
        'train_normal_n': TRAIN_NORMAL_N,
        'tune_normal_n': TUNE_NORMAL_N,
        'tune_defect_n': TUNE_DEFECT_N,
        'test_normal_n': TEST_NORMAL_N,
        'test_defect_n': TEST_DEFECT_N,
        'memory_bank_max': MEMORY_BANK_MAX_PATCHES,
        'score_chunk': SCORE_CHUNK,
        'patchcore_nn_k': PATCHCORE_NN_K,
        'topk_patch_ratio': TOPK_PATCH_RATIO,
    },
}
torch.save(artifact, MODEL_EXPORT_PATH)

metrics = {
    'roc_auc_z': roc_auc,
    'threshold_z': float(threshold_z),
    'threshold_raw': float(threshold_raw),
    'train_score_mu': float(mu),
    'train_score_std': float(std),
    'confusion_matrix': cm.tolist(),
    'n_test_normal': int(len(test_normal_scores_z)),
    'n_test_defect': int(len(test_defect_scores_z)),
    'run_training': bool(RUN_TRAINING),
    'model_export_path': MODEL_EXPORT_PATH,
    'scores_export_path': SCORES_EXPORT_PATH,
    'test_scores_csv_path': TEST_SCORES_CSV_PATH,
    'defect_recall_csv_path': DEFECT_RECALL_CSV_PATH,
    'classification_report_path': CLASSIFICATION_REPORT_PATH,
    'threshold_selection_png_path': THRESHOLD_SELECTION_PNG_PATH,
    'confusion_matrix_png_path': CONFUSION_MATRIX_PNG_PATH,
    'score_distribution_png_path': SCORE_DISTRIBUTION_PNG_PATH,
    'umap_png_path': UMAP_PNG_PATH,
    'umap_csv_path': UMAP_CSV_PATH,
}
Path(METRICS_EXPORT_PATH).write_text(json.dumps(metrics, indent=2), encoding='utf-8')

print(f'Saved model artifact: {MODEL_EXPORT_PATH}')
print(f'Saved metrics: {METRICS_EXPORT_PATH}')


## Cleanup


In [ ]:
# -- 13. Cleanup ---------------------------------------------------------------

for name in [
    'memory_bank',
    'memory_bank_t',
    'train_scores',
    'tune_normal_scores',
    'tune_defect_scores',
    'test_normal_scores',
    'test_defect_scores',
    'train_scores_z',
    'tune_normal_scores_z',
    'tune_defect_scores_z',
    'test_normal_scores_z',
    'test_defect_scores_z',
    'scores',
    'y_true',
    'y_pred',
    'train_loader',
    'tune_normal_loader',
    'tune_defect_loader',
    'test_normal_loader',
    'test_defect_loader',
]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print('Cleared in-memory tensors and dataloaders.')
